# Day 1: getting to know the data

**Python for Data Analysis**, Coding for Reproducible Research (CfRR), University of Exeter

## The question

Was summer 2026 actually unusual, or does it just feel that way? Over two sessions we answer it with the Met Office's monthly weather record for Oxford, which starts in 1853. Today we get the data into shape and take a first look. Tomorrow we plot it, fit a trend and bring in a second station.

## By the end of today you will be able to

- read a messy text file into a pandas DataFrame, choosing the separator, header and missing-value markers yourself
- explain how one stray character turns a column of numbers into text, and fix it without throwing information away
- summarise every column of a table at once with `.describe()`
- turn a column into dates, and pick rows by their label with `.loc` and by their position with `.iloc`
- pick out the rows and columns that answer a question, combining conditions with `&`, `|` and `~`
- decide what to do about missing data, and defend the decision
- summarise data with split-apply-combine (`groupby`), and check the groups before trusting the answer

## How to use this notebook

There are two versions of this notebook. In `day1.ipynb` the explanations are written for you and most code cells are empty: in the session, you type the code into them as the instructor types it on screen. A few cells, mostly setup, are filled in already; just run those. [`day1_solutions.ipynb`](day1_solutions.ipynb), in the same folder, has every cell filled in, with its output. If you fall behind, copy the cell you missed from there and carry on. Afterwards, either one is your notes.

The [CfRR Python for Data Analysis pages](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/section_landing_pages/python_for_data_analysis.html) are the full reference. This notebook links to them when you might want every option.

*Data: Met Office historic station data for Oxford. Contains public sector information licensed under the [Open Government Licence v3.0](https://www.nationalarchives.gov.uk/doc/open-government-licence/version/3/).*

## 1. Welcome, the question and the data

*10:00, 10 minutes*

Before any code: was summer 2026 in Oxford unusual?

### Meet the data

The Met Office publishes [monthly records](https://www.metoffice.gov.uk/research/climate/maps-and-data/historic-station-data) for a set of long-running UK weather stations: one plain text file per station, free to reuse. We will use Oxford, whose record starts in January 1853. The station is at 51.76°N, 1.26°W, 63 metres above sea level.

Each month in the file is summarised by five numbers:

| Column | What it measures | Units |
|---|---|---|
| `tmax` | Mean daily maximum temperature: each day's highest temperature, averaged over the month | °C |
| `tmin` | Mean daily minimum temperature: each night's lowest temperature, averaged over the month | °C |
| `af` | Days of air frost: days when the air temperature fell below 0 °C | days |
| `rain` | Total rainfall for the month | mm |
| `sun` | Total hours of sunshine in the month | hours |

Two things are easy to misread:

- **`tmax` is an average, not a record.** A July `tmax` of 22 °C means the daily highs averaged 22 °C. Some days were much hotter.
- **"Summer" means June, July and August.** That is the meteorologists' definition, and the one we will use.

The record is not perfect (data very rarely is!):

- **Estimated** values fill gaps using the Met Office's gridded dataset (HadUK-Grid) rather than a reading at the station.
- **Missing** values are months with more than two days of data missing.
- **Provisional** values have not yet been through the Met Office's full quality control, and may be revised.
- **Sunshine** was measured for most of the record by a Campbell-Stokes recorder, a glass ball that focuses the sun to burn a trace on a card. Some stations now use an automatic sensor instead.
- Over 170 years, the site and instruments may have changed slightly.

Knowing what each number means, and how it was made, is part of the analysis: it decides which questions the data can answer.

## 2. Python warm-up

*10:10, 10 minutes*

A quick reminder of the Python from the Introduction to Python course, before we need it. Every idea here comes back later today or tomorrow, on real data.

A notebook runs cells in the order you run them, not the order they appear on the page. Run a cell with Shift+Enter. If things get confusing, use **Kernel > Restart Kernel and Run All Cells**.

Every value has a **type**. Text is a string, `str`, and a string has **methods**: functions attached to it, called with a dot. Here is a temperature as it appears in today's file, with a marker on the end:

In [2]:
reading = "13.1*"

print(type(reading))
print(reading.endswith("*"))
print(reading.rstrip("*"))

<class 'str'>
True
13.1


`.endswith("*")` answers `True` or `False`, a `bool`. `.rstrip("*")` strips the `*` from the end, but the result is still text. `float` turns it into a number you can do arithmetic with. Comparisons give more `bool`s, and `and`, `or` and `not` combine them:

In [3]:
temperature = float(reading.rstrip("*"))
print(temperature + 1)
print(temperature > 10 and reading.endswith("*"))

14.1
True


A **list** holds values in order. Square brackets pick one out by its position, counting from 0, and a `for` loop visits each in turn:

In [4]:
months = ["Jun", "Jul", "Aug"]
print(months[0], len(months))
for month in months:
    print(month)

Jun 3
Jun
Jul
Aug


A **dictionary**, `dict`, looks values up by a key instead of a position:

In [5]:
station = {"name": "Oxford", "opened": 1853, "height_m": 63}
station["opened"]

1853

A **function** packages steps you want to reuse. `def` gives it a name and its inputs, the indented lines are its body, and `return` hands back the result:

In [6]:
def to_fahrenheit(celsius):
    return celsius * 9 / 5 + 32

to_fahrenheit(25)

77.0

You will meet all of these again today: a list of column names in block 4, string methods and a loop in block 5, and a dictionary and `and` in block 6. Tomorrow you will write a function that loads any station's file.

## 3. First look at the raw file

*10:20, 10 minutes*

### The tools

- **pandas** is the standard Python library for tables of data. Its central object is the **DataFrame**: rows and named columns, like a spreadsheet that you drive with code. Every step is written down, so anyone can rerun it and get the same result, which is hard to promise for clicks in a spreadsheet.
- **NumPy** does the fast arithmetic underneath pandas. We look at it properly tomorrow, before fitting a trend.
- **pathlib** builds file locations that work on Windows, macOS and Linux alike.

`import pandas as pd` is a convention: almost all the pandas code you will find elsewhere uses the same short name.

The data came in the zip with this notebook, in the `data` folder next to it. It is a dated snapshot of the Met Office file rather than the live one, so everyone works with exactly the same numbers, and they stay the same when the Met Office updates its own file. This first cell is filled in for you: run it. It should show `True`, meaning Python found the file.

In [7]:
from pathlib import Path

import pandas as pd

DATA = Path("data")
# Snapshot of https://www.metoffice.gov.uk/pub/data/weather/uk/climate/stationdata/oxforddata.txt
OXFORD = DATA / "oxforddata.txt"
OXFORD.exists()

True

Look at a data file before you load it. In the file browser on the left, open the `data` folder and double-click `oxforddata.txt`. JupyterLab opens it as plain text in a new tab. Use JupyterLab rather than Excel, which would try to split it into columns for you.

Read the top, then jump to the bottom with Ctrl+End (Cmd+Down on a Mac). What will make this file awkward to load into a table?

Six things about this file will make it awkward to load. How many did you find?

- five lines of free text before the column names
- a row of units under the column names
- columns separated by spaces rather than commas
- `---` for missing values
- a `*` after estimated values
- the word `Provisional` at the end of recent rows

Each one causes a problem in the next half hour. The obvious first try is `pd.read_csv` with no options at all:

In [8]:
df = pd.read_csv(OXFORD)
df.shape

(2091, 1)

In [9]:
df.head()

,,Oxford
Location: 450900E 207200N,Lat 51.761 Lon -1.262,63 metres amsl
Estimated data is marked with a * after the value.,nan,NaN
Missing data (more than 2 days missing in month) is marked by ---.,nan,NaN
Sunshine data taken from an automatic Kipp & Zonen sensor marked with a #,otherwise sunshine data taken from a Campbell Stokes recorder.,NaN
yyyy mm tmax tmin af rain sun,nan,NaN


One column. `read_csv` splits on commas by default and there are none, so each whole line becomes a single value, and the first line (`Oxford`) becomes the column name.

## 4. Loading it properly

*10:30, 15 minutes*

"CSV" means comma-separated values, but `read_csv` reads any delimited text once you tell it how. Reading a file is a set of decisions, not a default:

- **`sep`**: what separates the columns
- **`skiprows`**: how many lines to skip before the data
- **`names`**: what to call the columns
- **`na_values`**: how this file writes "missing"

The [CfRR Pandas page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Pandas.html) covers the other options.

First the separator. `sep=r"\s+"` means "one or more whitespace characters".

In [10]:
pd.read_csv(OXFORD, sep=r"\s+")

ParserError: Error tokenizing data. C error: Expected 10 fields in line 4, saw 13


An error. Read error messages from the bottom up: the last line says what went wrong, and everything above it says where. Here pandas expected 10 fields on line 4 and found 13. Line 4 is free text, not data.

So we skip all seven lines before the first row of numbers, and supply the column names ourselves. Remember the word `Provisional` at the end of recent rows: those rows have an eighth field, so the data are ragged, with some rows longer than others. That word tells us these numbers may still change, so rather than throw it away we give it a column of its own, `status`.

In [11]:
COLUMNS = ["yyyy", "mm", "tmax", "tmin", "af", "rain", "sun", "status"]
pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS)

,yyyy,mm,tmax,tmin,af,rain,sun,status
0,1853,1,8.4,2.7,4,62.8,---,NaN
1,1853,2,3.2,-1.8,19,29.3,---,NaN
2,1853,3,7.7,-0.6,20,25.9,---,NaN
3,1853,4,12.6,4.5,0,60.1,---,NaN
4,1853,5,16.8,6.1,0,59.5,---,NaN
...,...,...,...,...,...,...,...,...
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional


It loads, and the last rows show `Provisional` in `status`. Now look at `sun` in the first rows: `---` is this file's way of writing "missing", and pandas has kept it as text. `na_values` tells pandas to read it as missing instead:

In [12]:
df = pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS, na_values="---")
df.head()

,yyyy,mm,tmax,tmin,af,rain,sun,status
0,1853,1,8.4,2.7,4,62.8,NaN,NaN
1,1853,2,3.2,-1.8,19,29.3,NaN,NaN
2,1853,3,7.7,-0.6,20,25.9,NaN,NaN
3,1853,4,12.6,4.5,0,60.1,NaN,NaN
4,1853,5,16.8,6.1,0,59.5,NaN,NaN


In [13]:
df.tail()

,yyyy,mm,tmax,tmin,af,rain,sun,status
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional
2084,2026,9,22.2*,12.9*,0*,28.1*,149.1*,Provisional


### Anatomy of a DataFrame

Now the data are in, it is worth knowing the parts:

- the **columns**, each with a name
- the **index**: the row labels down the left (for now just 0, 1, 2 and so on)
- each column on its own is a **Series**: one column of values that keeps the index

`shape` gives the number of rows and columns, and `columns` their names. Selecting one column gives a Series: look at the last line of its output, `dtype`. We will need it soon.

In [14]:
df

,yyyy,mm,tmax,tmin,af,rain,sun,status
0,1853,1,8.4,2.7,4,62.8,NaN,NaN
1,1853,2,3.2,-1.8,19,29.3,NaN,NaN
2,1853,3,7.7,-0.6,20,25.9,NaN,NaN
3,1853,4,12.6,4.5,0,60.1,NaN,NaN
4,1853,5,16.8,6.1,0,59.5,NaN,NaN
...,...,...,...,...,...,...,...,...
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional


In [15]:
df.shape

(2085, 8)

In [16]:
df.columns

Index(['yyyy', 'mm', 'tmax', 'tmin', 'af', 'rain', 'sun', 'status'], dtype='str')

In [17]:
df["tmax"].head(3)

0    8.4
1    3.2
2    7.7
Name: tmax, dtype: str

`na_values="---"` turned the file's own missing marker into proper missing values (`NaN`). Count them in each column:

In [18]:
df.isna().sum()

yyyy         0
mm           0
tmax         0
tmin         1
af           1
rain         0
sun        912
status    2076
dtype: int64

`sun` has the most: sunshine was not recorded at Oxford until 1929. `status` is mostly missing too, but that is expected, because only provisional rows have a status. One month, December 1860, is missing `tmin` and `af`.

## 5. The messy data problem

*10:45, 20 minutes*

The first thing anyone wants from a column of temperatures is its average.

In [19]:
df["tmax"].mean()

TypeError: Cannot perform reduction 'mean' with string dtype

It fails. `.info()` summarises the whole DataFrame: every column, how many values it has (`Non-Null Count`) and its data type (`Dtype`):

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2085 entries, 0 to 2084
Data columns (total 8 columns):
 #   Column  Non-Null Count  Dtype
---  ------  --------------  -----
 0   yyyy    2085 non-null   int64
 1   mm      2085 non-null   int64
 2   tmax    2085 non-null   str  
 3   tmin    2084 non-null   str  
 4   af      2084 non-null   str  
 5   rain    2085 non-null   str  
 6   sun     1173 non-null   str  
 7   status  9 non-null      str  
dtypes: int64(2), str(6)
memory usage: 130.4 KB


`tmax` is stored as text (`str` in pandas 3, `object` in earlier versions). One value pandas cannot read as a number is enough to make the whole column text. Here are the culprits:

In [21]:
df[df["tmax"].str.endswith("*")].tail()

,yyyy,mm,tmax,tmin,af,rain,sun,status
2080,2026,5,20.7*,9.6*,0*,12.0,239.1*,Provisional
2081,2026,6,23.8*,14.4*,0*,58.7,231.9*,Provisional
2082,2026,7,28.2*,14.9*,0*,0.0,353.0*,Provisional
2083,2026,8,26.5*,14.3*,0*,28.4,267.9*,Provisional
2084,2026,9,22.2*,12.9*,0*,28.1*,149.1*,Provisional


The `*` marks an estimated value. That matters: July 2026, the month we care about, is one of them.

### The quick fix, and why it is wrong

`pd.to_numeric` converts text to numbers. On its own, it stops at the first value it cannot read:

In [22]:
pd.to_numeric(df["tmax"])

ValueError: Unable to parse string '13.1*' at position 1863

Ask an AI assistant, or search online, and you will often be told to add `errors="coerce"`. Try it:

In [23]:
suggested = pd.to_numeric(df["tmax"], errors="coerce")
suggested.mean().round(2)

np.float64(13.98)

No error, and an average that looks sensible. Now count the missing values it leaves:

In [24]:
suggested.isna().sum()

np.int64(26)

`errors="coerce"` turns anything it cannot read into `NaN`, without telling you. Every estimated value disappears, including July 2026. The code runs and the answer is wrong. That is the worst kind of bug.

A better approach keeps the information: record which values were estimated in a column of its own, then strip the `*` and convert. `.astype(float)` fails loudly if it meets anything else it does not expect, which is what we want. `.str.rstrip("*")` strips a `*` from the end of each value, where this file puts it; `.str.replace("*", "")` would remove one from anywhere, and works here too.

In [25]:
# Keep the flag, then strip the marker and convert
estimated = df["tmax"].str.endswith("*")
estimated.sum()

np.int64(26)

Now the mean includes every month. Compare it with the 13.98 °C from `errors="coerce"`:

In [26]:
tmax_clean = df["tmax"].str.rstrip("*").astype(float)
tmax_clean.mean().round(2)

np.float64(14.03)

Higher. Most estimated values are recent, warm months, so dropping them quietly pulled the average down.

All five measured columns need the same treatment, and the results should go back into `df`. A loop does it once for all of them: it replaces each column with its cleaned version and adds an `_estimated` column alongside it.

`astype(str)` first makes sure every column is text before we look for a `*`. At some stations a column has no estimates at all, so pandas reads it as numbers already, and `.str` only works on text. The last line, `df.dtypes`, lists just the data types, so we can check that the five measurements are now numbers (`float64`).

Run the loop only once. It changes `df` itself, so a second run finds no `*` left and quietly sets every `_estimated` column to `False`. If you do run it twice, run the `df = pd.read_csv(...)` cell in block 4 again, then the loop, and carry on from there.

In [27]:
MEASURES = ["tmax", "tmin", "af", "rain", "sun"]
for col in MEASURES:
    text = df[col].astype(str)
    df[col + "_estimated"] = text.str.endswith("*")
    df[col] = text.str.rstrip("*").astype(float)
df.dtypes

yyyy                int64
mm                  int64
tmax              float64
tmin              float64
af                float64
rain              float64
sun               float64
status                str
tmax_estimated       bool
tmin_estimated       bool
af_estimated         bool
rain_estimated       bool
sun_estimated        bool
dtype: object

The `status` column gets the same idea: a `True`/`False` column called `provisional` keeps the information, and then `status` can go.

In [28]:
df["provisional"] = df["status"] == "Provisional"
df = df.drop(columns="status")
df[["yyyy", "mm", "tmax", "tmax_estimated", "provisional"]].tail(3)

,yyyy,mm,tmax,tmax_estimated,provisional
2082,2026,7,28.2,True,True
2083,2026,8,26.5,True,True
2084,2026,9,22.2,True,True


We have kept every flag as its own column rather than deleting the estimated and provisional rows. Throwing them away would have thrown away July 2026.

### Summarising every column

Now that every measurement is a number, `.describe()` summarises them all in one table: how many values each column has (`count`), the mean and the standard deviation (`std`), the smallest and largest values, and the quartiles in between. `.round(1)` keeps it readable.

In [29]:
df[MEASURES].describe().round(1)

,tmax,tmin,af,rain,sun
count,2085.0,2084.0,2084.0,2085.0,1173.0
mean,14.0,6.3,3.7,54.9,129.5
std,5.7,4.2,5.3,31.7,63.9
min,-0.2,-5.8,0.0,0.0,18.2
25%,9.1,2.9,0.0,31.3,72.6
50%,13.9,5.7,1.0,49.9,123.5
75%,19.1,10.2,6.0,74.8,176.1
max,28.2,15.7,28.0,197.1,353.0


A lot can be read off this one table:

- **count** shows the gaps: `sun` has far fewer values than the other columns, and `tmin` and `af` are one short.
- **min** and **max** are a quick check that nothing is impossible. The hottest month, 28.2 °C, is July 2026.
- **50%** is the median: in half of all months, less than 49.9 mm of rain fell.

`.describe()` is worth running on any dataset as soon as it loads properly.

## Break

*11:05, 10 minutes*

## 6. Picking rows: dates, labels and masks

*11:15, 25 minutes*

Most questions about data start with "which rows and columns?": which month was the hottest, which winters had no frost. pandas has three ways to pick rows: by their **label** in the index, by their **position**, and with a **mask** that tests every row. Labels are only useful if they mean something, so first give every row a proper date.

In [30]:
df.iloc[102:][['yyyy', 'mm']]

,yyyy,mm
102,1861,7
103,1861,8
104,1861,9
105,1861,10
106,1861,11
...,...,...
2080,2026,5
2081,2026,6
2082,2026,7
2083,2026,8


In [31]:
df.loc[0:100, ('yyyy', 'mm')]

,yyyy,mm
0,1853,1
1,1853,2
2,1853,3
3,1853,4
4,1853,5
...,...,...
96,1861,1
97,1861,2
98,1861,3
99,1861,4


### A date for every row

`pd.to_datetime` makes dates. Given a dictionary of years, months and days, it builds one date for each row. Our rows are whole months, so use the 1st as the day:

In [32]:
df["date"] = pd.to_datetime({"year": df["yyyy"], "month": df["mm"], "day": 1})
df["date"].head()

0   1853-01-01
1   1853-02-01
2   1853-03-01
3   1853-04-01
4   1853-05-01
Name: date, dtype: datetime64[us]

The last line of the output says `datetime64`: pandas' type for dates and times. `yyyy` and `mm` are just numbers, but a `datetime64` column knows it holds dates. The `.dt` accessor reaches the parts of each date, as `.str` reached into text:

In [33]:
df["date"].dt.weekday.tail()
#df["date"].dt.month_name().tail(3)

/var/folders/d_/2jd5klgs5wzcvr9c29w51qr40000gp/T/ipykernel_73193/3780465114.py:1: Pandas4Warning: Series.dt.weekday is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  df["date"].dt.weekday.tail()


2080    4
2081    0
2082    2
2083    5
2084    1
Name: date, dtype: int32

Dates in most files arrive as text, such as `01/07/2026`. `pd.to_datetime` reads text too, but is that 1 July or 7 January? Unless told, pandas assumes the American order, month first. `format` says exactly how to read it: `%d` is the day, `%m` the month and `%Y` the four-digit year.

In [34]:
print(pd.to_datetime("01/07/2026")) 
print(pd.to_datetime("01/07/2026", format="%d/%m/%Y"))

2026-01-07 00:00:00
2026-07-01 00:00:00


Without `format`, 7 January, and no error. Whenever dates arrive as text, give `format`: pandas then reads them the way you mean, and fails loudly on any value that doesn't match. The [CfRR Pandas page](https://coding-for-reproducible-research.github.io/CfRR_Courses/individual_modules/python_for_data_analysis/Python_Pandas.html) uses it on dates with times.

### Dates as the index

`set_index` moves a column into the index, so the dates become the row labels down the left. `reset_index` does the reverse. Run the next cell only once: afterwards `date` is the index rather than a column, so a second run gives a `KeyError`. If that happens, run the cell that made `df["date"]` again, then this one.

In [35]:
df = df.set_index("date")
df[["yyyy", "mm", "tmax"]].head(3)

,yyyy,mm,tmax
date,,,
1853-01-01,1853,1,8.4
1853-02-01,1853,2,3.2
1853-03-01,1853,3,7.7


`.loc` picks rows by their label. With dates as labels, you can write a month (`"2026-07"`), a year (`"1976"`) or a range of dates as text. `.loc[rows, columns]` takes the rows first, then the columns: one name, or a list of names. A list has its own square brackets, which is why you have already seen double brackets, as in `df[["yyyy", "mm", "tmax"]]`. Summer 1976, a famous heatwave, with two of its columns:

In [36]:
df.loc["1976-06":"1976-08", ["tmax", "rain"]]

,tmax,rain
date,,
1976-06-01,24.4,17.1
1976-07-01,25.9,14.3
1976-08-01,24.4,23.9


Unlike a Python slice, a `.loc` range includes its last label, so August is in. `.iloc` ignores the labels and counts positions from 0, like a list, so `df.iloc[0]` is the first row whatever its label:

In [37]:
df.iloc[0:100]

,yyyy,mm,tmax,tmin,af,rain,sun,tmax_estimated,tmin_estimated,af_estimated,rain_estimated,sun_estimated,provisional
date,,,,,,,,,,,,,
1853-01-01,1853,1,8.4,2.7,4.0,62.8,NaN,False,False,False,False,False,False
1853-02-01,1853,2,3.2,-1.8,19.0,29.3,NaN,False,False,False,False,False,False
1853-03-01,1853,3,7.7,-0.6,20.0,25.9,NaN,False,False,False,False,False,False
1853-04-01,1853,4,12.6,4.5,0.0,60.1,NaN,False,False,False,False,False,False
1853-05-01,1853,5,16.8,6.1,0.0,59.5,NaN,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1860-12-01,1860,12,4.4,NaN,NaN,66.6,NaN,False,False,False,False,False,False
1861-01-01,1861,1,3.8,-0.7,16.0,16.8,NaN,False,False,False,False,False,False
1861-02-01,1861,2,7.9,3.1,4.0,48.0,NaN,False,False,False,False,False,False


January 1853. Before `set_index` the labels were 0, 1, 2 and so on, so `df.loc[0]` and `df.iloc[0]` were the same row. Now `df.loc[0]` fails: no row is labelled 0 any more.

### Masks: one condition

A **boolean mask** picks rows by testing every one of them: a column of `True`/`False` values, one for each row, saying whether that row passes a test. You have made masks already: `df["tmax"].str.endswith("*")` in block 5 was one, and each `_estimated` column is one kept in the DataFrame. After the second break you will practise with masks, and everything else from today, on your own.

The simplest mask is one comparison on a column. When you add up a mask, `True` counts as 1, so `.sum()` counts the rows that pass:

In [38]:
frost_free = df["af"] == 0
print(frost_free.sum())
frost_free

1009


date
1853-01-01    False
1853-02-01    False
1853-03-01    False
1853-04-01     True
1853-05-01     True
              ...  
2026-05-01     True
2026-06-01     True
2026-07-01     True
2026-08-01     True
2026-09-01     True
Name: af, Length: 2085, dtype: bool

For the same reason, `.mean()` of a mask is the fraction of rows that pass:

In [39]:
frost_free.mean().round(2)

np.float64(0.48)

Just under half of all months had no air frost.

### Rows and columns together

As with summer 1976, `df.loc[rows, columns]` picks rows and columns at once, and the rows can be a mask:

In [40]:
df.loc[frost_free, ["yyyy", "mm", "tmax"]].head()

,yyyy,mm,tmax
date,,,
1853-04-01,1853,4,12.6
1853-05-01,1853,5,16.8
1853-06-01,1853,6,20.1
1853-07-01,1853,7,21.2
1853-08-01,1853,8,20.2


In [41]:
df.loc[frost_free, ["tmax"]].median()

tmax    19.2
dtype: float64

### Combining conditions

Python's own `and`, `or` and `not` don't work on whole columns of `True`/`False`. pandas uses three symbols instead:

- `&` (and): true where both masks are true
- `|` (or): true where at least one is true
- `~` (not): flips `True` and `False`

Which hot months are estimated? Put each comparison in its own brackets:

In [42]:
hot_estimated = (df["tmax"] > 23) & df["tmax_estimated"]
df.loc[hot_estimated, ["yyyy", "mm", "tmax"]]

,yyyy,mm,tmax
date,,,
2025-06-01,2025,6,24.0
2025-07-01,2025,7,25.3
2026-06-01,2026,6,23.8
2026-07-01,2026,7,28.2
2026-08-01,2026,8,26.5


Five months, all from the last two summers: the hottest months in the record are also the least certain.

Now the same question with the brackets left out:

In [43]:
len(df[df["tmax"] > 23 & df["tmax_estimated"]])

2083

2083 of the 2085 months, and no error. Python works out `&` before `>`, so this compares `tmax` with `23 & df["tmax_estimated"]`, which is 1 or 0, and almost every month is warmer than that. Like `errors="coerce"`, it runs and the answer is wrong. Put every comparison in brackets, and check that the number of rows makes sense.

`|` (or) works just like `&`, with brackets round each comparison; you will use it in the practice. `~` flips a mask, so `~df["tmax_estimated"]` is every month that was measured rather than estimated. The hottest of those:

In [44]:
df.loc[(df["tmax"] > 26) | (df["tmin"] < -3)]
df.loc[(df["tmax"] > 26) | (df["tmin"] < -3), ["yyyy", "mm", "tmax", "tmin"]]

,yyyy,mm,tmax,tmin
date,,,,
1855-02-01,1855,2,1.7,-4.5
1879-12-01,1879,12,3.7,-3.2
1881-01-01,1881,1,1.8,-4.0
1890-12-01,1890,12,0.2,-4.0
1895-02-01,1895,2,1.8,-5.3
1911-07-01,1911,7,26.1,13.1
1921-07-01,1921,7,26.1,13.3
1940-01-01,1940,1,1.9,-4.6
1947-02-01,1947,2,-0.2,-4.4


In [45]:
df.loc[~df["tmax_estimated"], "tmax"].max()

np.float64(27.4)

27.4 °C, July 2018: the record if you set the estimates aside.

### A shortcut: `.isin()`

`.isin()` is true where a value is one of a list, so `df["mm"].isin([12, 1, 2])` does the job of three `==` comparisons joined with `|`. It is a method rather than a comparison, so it needs no extra brackets. Winter months with an average maximum above 11 °C:

In [46]:
winter = df["mm"].isin([12, 1, 2])
df.loc[winter & (df["tmax"] > 11), ["yyyy", "mm", "tmax"]]

,yyyy,mm,tmax
date,,,
1990-02-01,1990,2,11.3
1998-02-01,1998,2,11.5
2002-02-01,2002,2,11.1
2015-12-01,2015,12,13.4
2019-02-01,2019,2,11.8
2022-02-01,2022,2,11.1
2024-02-01,2024,2,11.4


All seven are from 1990 onwards.

## 7. Missing data is a decision

*11:40, 10 minutes*

`df.dropna()` removes every row with a missing value; `df.dropna(subset=["tmin"])` would only look at the columns you list. How many rows would survive?

In [47]:
len(df), len(df.dropna())

(2085, 1173)

Only just over half. Sunshine is missing for every month from 1853 to 1928, so dropping those rows also throws away 76 years of perfectly good temperature records.

There are five things we could do about the missing sunshine, and each has a cost:

1. **Drop every row with a missing value.** Loses most of the temperature record from before 1929.
2. **Drop the `sun` column.** Loses the sunshine we do have.
3. **Fill the gaps with the average sunshine.** Invents data, as the next cell shows.
4. **Fill the gaps by linear interpolation:** a straight line from the reading before each gap to the reading after it. Needs a reading on both sides.
5. **Keep everything, leave the gaps as `NaN`, and only use sunshine from 1929.**

We take option 5. pandas skips `NaN` when it calculates, so the gaps do no harm as long as we know they are there. `.count()` counts the values that are not missing, and `.mean()` averages only those:

In [48]:
df["sun"].count(), df["sun"].mean().round(1)

(np.int64(1173), np.float64(129.5))

1173 months have sunshine, and the mean is taken over those alone. Option 3 is the most dangerous, because it looks harmless:

In [49]:
filled = df["sun"].fillna(df["sun"].mean())
df["sun"].std(), filled.std()

(np.float64(63.86273077602531), np.float64(47.89194042558409))

Filling invents 76 years of perfectly average sunshine, and makes the record look far less variable than it was. Any analysis of sunshine over time would be quietly wrong.

Option 4 sounds more careful. `.interpolate()` fills each gap with a straight line between the readings either side. How many sunshine values are still missing afterwards?

In [50]:
df["sun"].interpolate().isna().sum()

np.int64(912)

All 912 of them. Every missing month comes before the first sunshine reading, in January 1929, so there is no reading before the gap to start the line from. Interpolation suits short gaps inside a record, like the single missing `tmin` in December 1860:

In [51]:
df["tmin"].loc["1860-10":"1861-02"]

date
1860-10-01    7.3
1860-11-01    2.1
1860-12-01    NaN
1861-01-01   -0.7
1861-02-01    3.1
Name: tmin, dtype: float64

In [52]:
df["tmin"].interpolate().loc["1860-10":"1861-02"]

date
1860-10-01    7.3
1860-11-01    2.1
1860-12-01    0.7
1861-01-01   -0.7
1861-02-01    3.1
Name: tmin, dtype: float64

December 1860 gets 0.7 °C, halfway between November and January. Filling one month in a long record like this is defensible, as long as you say so. We haven't saved it back into `df`, so `tmin` keeps its one `NaN`.

## 8. Summarising: split-apply-combine

*11:50, 25 minutes*

Before writing code, say the plan in plain English. To get the average summer (June to August) maximum temperature for each year:

1. keep only the June, July and August rows
2. **split** them into groups by year
3. **apply** the mean to `tmax` in each group
4. **combine** the results into one table

That is split-apply-combine, and `groupby` does the last three steps in one line.

### One group by hand

You can already do this for one year with masks. Keep the summer rows, then average 2026's `tmax`:

In [53]:
summer = df[df["mm"].isin([6, 7, 8])]
summer.loc[summer["yyyy"] == 2026, "tmax"].mean()

np.float64(26.166666666666668)

In [54]:
summer

,yyyy,mm,tmax,tmin,af,rain,sun,tmax_estimated,tmin_estimated,af_estimated,rain_estimated,sun_estimated,provisional
date,,,,,,,,,,,,,
1853-06-01,1853,6,20.1,10.7,0.0,82.0,NaN,False,False,False,False,False,False
1853-07-01,1853,7,21.2,12.2,0.0,86.2,NaN,False,False,False,False,False,False
1853-08-01,1853,8,20.2,10.8,0.0,72.3,NaN,False,False,False,False,False,False
1854-06-01,1854,6,18.6,9.7,0.0,45.2,NaN,False,False,False,False,False,False
1854-07-01,1854,7,21.7,11.6,0.0,37.2,NaN,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-07-01,2025,7,25.3,14.6,0.0,40.7,217.8,True,False,False,False,False,False
2025-08-01,2025,8,24.2,13.4,0.0,23.5,205.2,False,True,True,False,False,False
2026-06-01,2026,6,23.8,14.4,0.0,58.7,231.9,True,True,True,False,True,True


That is one group. There are 174 summers in the record, and doing each one like this would take a loop over the years. `groupby` does every group at once.

### What `groupby` gives back

Step 2, the split, on its own:

In [55]:
means = {}
for year in summer["yyyy"].unique():
    means[year] = summer.loc[summer["yyyy"] == year, "tmax"].mean()
pd.Series(means)

1853    20.500000
1854    20.400000
1855    21.300000
1856    21.700000
1857    22.266667
          ...    
2022    24.666667
2023    22.266667
2024    21.966667
2025    24.500000
2026    26.166667
Length: 174, dtype: float64

In [56]:
summer.groupby("yyyy")

Not a table: a `DataFrameGroupBy` object, the rows sorted into one group per year, waiting for something to apply. Nothing has been calculated yet.

Before applying anything, check the groups are what you expect. `.size()` counts the rows in each:

In [57]:
summer.groupby("yyyy").size()

yyyy
1853    3
1854    3
1855    3
1856    3
1857    3
       ..
2022    3
2023    3
2024    3
2025    3
2026    3
Length: 174, dtype: int64

Three months in every summer, 2026 included, so no summer's average rests on a single month.

### Apply and combine

Now steps 3 and 4: pick the column to work on, `["tmax"]`, and apply `.mean()` to it in each group. pandas combines the answers into one Series, with one value per group:

In [58]:
summer_tmax = summer.groupby("yyyy")["tmax"].mean()
summer_tmax.tail()

yyyy
2022    24.666667
2023    22.266667
2024    21.966667
2025    24.500000
2026    26.166667
Name: tmax, dtype: float64

The last value is the one we worked out by hand. The groups become the index: the years label the rows, so `.loc` picks out a year, as it picked out a date in block 6, and `.sort_values()` will rank them.

### Different groups, different questions

Group by month instead of year, using all the rows, and the same line answers a different question:

In [59]:
df.groupby("mm")["tmax"].mean()

mm
1      6.827011
2      7.586782
3     10.123563
4     13.330460
5     16.853448
6     20.054598
7     21.959770
8     21.400575
9     18.658621
10    14.270520
11     9.778613
12     7.444509
Name: tmax, dtype: float64

In [60]:
df["since_1990"] = df["yyyy"] >= 1990
result = df.groupby(["mm", "since_1990"])["tmax"].mean().round(1)

In [61]:
result.reset_index()

,mm,since_1990,tmax
0,1,False,6.5
1,1,True,8.0
2,2,False,7.2
3,2,True,8.9
4,3,False,9.8
5,3,True,11.5
6,4,False,13.0
7,4,True,14.5
8,5,False,16.6
9,5,True,17.9


This is the average maximum temperature for each month of the year over the whole record: the seasonal cycle. `.agg()` applies several functions at once, one column for each. Here is the cycle with its extremes:

In [62]:
df.groupby("mm")["tmax"].agg(["mean", "min", "max"]).round(1)

,mean,min,max
mm,,,
1,6.8,-0.2,10.4
2,7.6,-0.2,11.8
3,10.1,6.0,15.5
4,13.3,9.5,19.4
5,16.9,12.9,20.7
6,20.1,15.9,24.4
7,22.0,17.5,28.2
8,21.4,17.1,26.5
9,18.7,15.2,23.2


Every July's mean maximum since 1853 lies between 17.5 °C and 28.2 °C. `.count()` counts the values in each group that are not missing:

In [63]:
df.groupby("mm")["sun"].count()

mm
1     98
2     98
3     98
4     98
5     98
6     98
7     98
8     98
9     98
10    97
11    97
12    97
Name: sun, dtype: int64

174 years of temperatures, but sunshine for only 98 of them, because it starts in 1929. October to December have 97, because the record stops in September 2026. As in block 7, the missing values are left out, not counted as zero.

Now rank every July in the record by `tmax`, hottest first, and show the top five with their flags:

In [64]:
julys = df[df["mm"] == 7].sort_values("tmax", ascending=False)
julys[["tmax", "tmax_estimated", "provisional"]].head()

,tmax,tmax_estimated,provisional
date,,,
2026-07-01,28.2,True,True
2018-07-01,27.4,False,False
2006-07-01,27.1,False,False
1983-07-01,26.8,False,False
1911-07-01,26.1,False,False


## 9. Cliffhanger

*12:15, 5 minutes*

Where does summer 2026 come in the ranking of whole summers?

In [65]:
summer_tmax.sort_values(ascending=False).head()

yyyy
2026    26.166667
1976    24.900000
2022    24.666667
2018    24.533333
2025    24.500000
Name: tmax, dtype: float64

So July 2026 is the hottest July on record, and summer 2026 the hottest summer. Is that enough to say it was unusual?

- The 2026 temperatures are estimated and provisional.
- In any long record, some year has to be the hottest. Is 2026 far ahead, or just a bit ahead?
- Summers have been getting warmer. Is 2026 unusual for its time, or just the latest step on a trend?

Tomorrow we plot it, fit a trend and find out.

### Today's ideas

- Reading a file is a set of decisions: `sep`, `skiprows`, `names`, `na_values`.
- One stray character turns a numeric column into text. Fix it and keep the information as a flag.
- `errors="coerce"` hides problems. Prefer code that fails loudly.
- `.describe()` summarises every column in one table. Run it on any new dataset.
- `pd.to_datetime` makes real dates; give it a `format` when dates are text. With dates as the index, `.loc` picks rows by label and `.iloc` by position.
- A mask is a column of `True`/`False` that picks rows. Combine masks with `&`, `|` and `~`, with every comparison in brackets.
- pandas skips `NaN` when it calculates. `.count()` tells you how many values it used.
- Dropping or filling missing data is a decision with consequences.
- `groupby` is split-apply-combine. On its own it only splits: check the groups with `.size()`, then apply `.mean()`, `.agg()` or `.count()`.

## Break

*12:20, 10 minutes*

## 10. Practice

*12:30, 30 minutes*

Time to use everything from today on your own. The questions are grouped under the parts of the day they practise. Try at least one under each heading, then as many more as you like, in any order. There are more than you will finish today: carry on with the rest in your own time.

Write each answer in the empty cell under its question. The answers are in `day1_solutions.ipynb`, but try each one yourself first. If a mask is not doing what you expect, run it on its own and check its `.sum()`. Make every mask from `df` as it is now, such as `df["tmax_estimated"]`: masks made before `set_index`, like `estimated` in block 5, still have the old labels and no longer line up with `df`.

### Loading the file

**1.** `read_csv` has an option, `nrows`, that reads only that many rows of data. Read just the first two years (24 months) of the Oxford file into `first_two`, with the same separator, skipped lines, column names and missing-value marker as this morning. What is the dtype of its `tmax` column, and why isn't it text this time?

In [66]:
first_two = pd.read_csv(OXFORD, sep=r"\s+", skiprows=7, names=COLUMNS,
                        na_values="---", nrows=24)
first_two["tmax"].dtype

dtype('float64')

### Cleaning and summarising

**2.** How many values in each measurement column are estimated? Pick out the five `_estimated` columns with a list of names, then use `.sum()`.

In [67]:
df[["tmax_estimated", "tmin_estimated", "af_estimated", "rain_estimated",
    "sun_estimated"]].sum()

tmax_estimated    26
tmin_estimated    22
af_estimated      22
rain_estimated    21
sun_estimated     28
dtype: int64

**3.** `.rename(columns={...})` gives columns new names, with a dictionary from each old name to its new one. Make `readable`, a copy of `df` with `af` renamed to `frost_days` and `sun` to `sunshine_hours`, and show the last three rows of those two columns. Does `df` still have a column called `af`?

In [68]:
readable = df.rename(columns={"af": "frost_days", "sun": "sunshine_hours"})
readable[["frost_days", "sunshine_hours"]].tail(3)

,frost_days,sunshine_hours
date,,
2026-07-01,0.0,353.0
2026-08-01,0.0,267.9
2026-09-01,0.0,149.1


In [69]:
"af" in df.columns

True

**4.** Use `.describe()` to compare `rain` in summer (June to August) with winter (December to February). Which season is wetter on average, and which varies more?

In [70]:
SEASONS = {12: "winter", 1: "winter", 2: "winter",
           3: "spring", 4: "spring", 5: "spring",
           6: "summer", 7: "summer", 8: "summer",
           9: "autumn", 10: "autumn", 11: "autumn"}
df["season"] = df["mm"].map(SEASONS)

In [71]:
df.groupby('season')['rain'].describe().round(1)

,count,mean,std,min,25%,50%,75%,max
season,,,,,,,,
autumn,520.0,61.9,35.5,2.6,35.1,55.1,82.2,197.1
spring,522.0,47.2,27.0,0.5,27.2,44.6,60.4,143.0
summer,522.0,56.7,31.9,0.0,32.3,52.6,80.3,155.1
winter,521.0,53.7,30.0,2.2,30.1,50.7,74.2,147.9


In [72]:
df.loc[df["mm"].isin([6, 7, 8]), "rain"].describe().round(1)

count    522.0
mean      56.7
std       31.9
min        0.0
25%       32.3
50%       52.6
75%       80.3
max      155.1
Name: rain, dtype: float64

In [73]:
df.loc[df["mm"].isin([12, 1, 2]), "rain"].describe().round(1)

count    521.0
mean      53.7
std       30.0
min        2.2
25%       30.1
50%       50.7
75%       74.2
max      147.9
Name: rain, dtype: float64

### Dates and the index

**5.** Use `.loc` and a date to show `tmax`, `rain` and `sun` for January 2000, or for the month you were born. Then use `.iloc` to find the 1000th month in the record. Which month is it?

In [74]:
df.loc["1994-09", ["tmax", "rain", "sun"]]

,tmax,rain,sun
date,,,
1994-09-01,16.6,62.6,113.1


In [75]:
df.iloc[999]

yyyy                1936
mm                     4
tmax                10.7
tmin                 2.9
af                   4.0
rain                41.8
sun                133.2
tmax_estimated     False
tmin_estimated     False
af_estimated       False
rain_estimated     False
sun_estimated      False
provisional        False
since_1990         False
season            spring
Name: 1936-04-01 00:00:00, dtype: object

**6.** The winter of 1962 to 1963, the "Big Freeze", was one of the coldest in English history. Use a `.loc` range of dates to pick out December 1962 to February 1963. What was its mean `tmax`, and how many days of air frost did it have in total?

In [76]:
big_freeze = df.loc["1962-12":"1963-02"]
big_freeze["tmax"].mean().round(1), big_freeze["af"].sum()

(np.float64(1.9), np.float64(70.0))

### Masks

**7.** Make a DataFrame of the months when `tmin` was below 0 °C, with only the `yyyy` and `sun` columns. How many months are there, and why is `sun` missing for so many of them?

In [77]:
below_zero = df.loc[df["tmin"] < 0, ["yyyy", "sun"]]
below_zero

,yyyy,sun
date,,
1853-02-01,1853,NaN
1853-03-01,1853,NaN
1853-12-01,1853,NaN
1855-01-01,1855,NaN
1855-02-01,1855,NaN
...,...,...
1991-02-01,1991,60.0
1996-02-01,1996,103.4
1997-01-01,1997,51.4


**8.** What was the median rainfall in months when `tmax` was above 20 °C? How does it compare with the median for all months?

In [78]:
print(df.loc[df["tmax"] > 20, "rain"].median().round(1))
print(df["rain"].median().round(1))

43.3
49.9


**9.** Hot and dry: how many months had a `tmax` above 22 °C and less than 20 mm of rain? How many of them were in 2000 or later?

In [79]:
hot_dry = (df["tmax"] > 22) & (df["rain"] < 20)
print(hot_dry.sum())
print((hot_dry & (df["yyyy"] >= 2000)).sum())

39
9


**10.** How many months were either very wet (more than 150 mm of rain) or very dry (less than 5 mm)? Combine the two conditions with `|`.

In [80]:
((df["rain"] > 150) | (df["rain"] < 5)).sum()

np.int64(45)

**11.** Climate scientists compare each month with its average over 30 years, often 1991 to 2020. `.between(low, high)` is true where a value is in a range, both ends included. Use it to find the mean July `tmax` from 1991 to 2020. How far above it was July 2026, at 28.2 °C?

In [81]:
july_normal = df.loc[df["yyyy"].between(1991, 2020) & (df["mm"] == 7), "tmax"].mean()
july_normal.round(1), (28.2 - july_normal).round(1)

(np.float64(23.0), np.float64(5.1))

### Whole columns and missing values

**12.** Convert `tmax` to degrees Fahrenheit for every month at once, without a loop. The warm-up's `to_fahrenheit` works on a whole column too: try it. What is the hottest month in the record in °F?

In [82]:
tmax_f = to_fahrenheit(df["tmax"])
tmax_f.max().round(1)

np.float64(82.8)

**13.** In which year does the sunshine record start? Use `~` and `.isna()` to keep only the months with sunshine, then find the smallest `yyyy`.

In [83]:
df.loc[~df["sun"].isna(), "yyyy"].min()

np.int64(1929)

### Split-apply-combine

**14.** Which month of the year is the wettest on average, and which is the driest?

In [84]:
df.groupby("mm")["rain"].mean().round(1) #.idxmin()

mm
1     56.6
2     43.7
3     44.6
4     44.4
5     52.6
6     53.5
7     57.1
8     59.4
9     56.3
10    67.9
11    61.6
12    60.9
Name: rain, dtype: float64

**15.** Which five years had the most days of air frost in total?

In [85]:
df.groupby("yyyy")["af"].sum().sort_values(ascending=False).head()

yyyy
1879    93.0
1892    90.0
1855    82.0
1929    81.0
1887    81.0
Name: af, dtype: float64

### Your own question

**16.** Write a question of your own about the data that needs at least two conditions, and answer it. The solutions notebook has one example.

In [86]:
# One example: has there ever been an air frost between June and September?
df.loc[df["mm"].between(6, 9) & (df["af"] > 0), ["yyyy", "mm", "af"]]

,yyyy,mm,af
date,,,
1919-09-01,1919,9,1.0
